# Module 1: Comparison of Memory-Based and Generator-Based Data Loading

> **Assignment (10 points):** Compare loading all satellite images into memory vs. loading them lazily with a generator.

Data layout used by all labs:

```
images_dataSAT/
├── class_0_non_agri/   (non-agricultural land)
└── class_1_agri/       (agricultural land)
```

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

dir_non_agri = './images_dataSAT/class_0_non_agri/'
dir_agri     = './images_dataSAT/class_1_agri/'

def get_image_paths(directory):
    return sorted([os.path.join(directory, f) for f in os.listdir(directory)
                   if os.path.isfile(os.path.join(directory, f))])

## Memory-based loading: load **all** images into RAM at once

In [ ]:
def load_all_images(directory):
    paths = get_image_paths(directory)
    images = np.array([np.array(Image.open(p).resize((64, 64))) for p in paths])
    return paths, images

non_agri_paths, non_agri_images = load_all_images(dir_non_agri)
agri_paths,     agri_images     = load_all_images(dir_agri)

image_data   = np.concatenate([non_agri_images, agri_images])
image_labels = np.array([0] * len(non_agri_images) + [1] * len(agri_images))

print('image_data shape :', image_data.shape)
print('image_data memory: %.1f MB' % (image_data.nbytes / 1e6))

### Task 1: Determine the shape (dimensions) of a single image stored in the `image_data` variable

In [ ]:
single_image = image_data[0]
print('Shape of a single image:', single_image.shape)
print('Data type              :', single_image.dtype)

### Task 2: Display the first four images in `'./images_dataSAT/class_0_non_agri/'`

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(non_agri_images[i])
    axes[i].set_title(f'Non-Agri image {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

### Task 3: Create `agri_images_paths` — full paths of all images in `dir_agri`, **sorted**

In [ ]:
agri_images_paths = [os.path.join(dir_agri, f) for f in os.listdir(dir_agri)
                    if os.path.isfile(os.path.join(dir_agri, f))]
agri_images_paths.sort()

print('Number of paths:', len(agri_images_paths))
print('First 5 paths  :')
for p in agri_images_paths[:5]:
    print(' ', p)

### Task 4: Number of agricultural land images in `'./images_dataSAT/class_1_agri/'`

In [ ]:
num_agri_images = len(agri_images_paths)
print(f'Number of agricultural land images: {num_agri_images}')

### Task 5: Display the first four agricultural land images

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = np.array(Image.open(agri_images_paths[i]).resize((64, 64)))
    axes[i].imshow(img)
    axes[i].set_title(f'Agri image {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Generator-based loading: lazy, one batch at a time

In [ ]:
def custom_data_generator(list_of_paths, labels, batch_size=8, image_size=(64, 64), shuffle=True):
    indices = list(range(len(list_of_paths)))
    while True:
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start:start + batch_size]
            batch_images = np.array([np.array(Image.open(list_of_paths[i]).resize(image_size)) for i in batch_idx])
            batch_labels = np.array([labels[i] for i in batch_idx])
            yield batch_images, batch_labels

all_paths = agri_paths + non_agri_paths
all_path_labels = [1] * len(agri_paths) + [0] * len(non_agri_paths)

gen = custom_data_generator(all_paths, all_path_labels, batch_size=8)
X_batch, y_batch = next(gen)
print('One generator batch shape:', X_batch.shape)
print('Batch labels             :', y_batch)

In [ ]:
import sys
print('Full in-memory array : %.1f MB' % (image_data.nbytes / 1e6))
print('Generator object     : %d bytes (lazy, loads only one batch at a time)' % sys.getsizeof(gen))

> **Conclusion:** memory-based loading loads the whole dataset into RAM at once (here ~95 MB, and it grows linearly with dataset size), while generator-based loading reads and preprocesses one small batch at a time, keeping memory usage flat — the memory-efficient strategy for large geospatial datasets.